# WP1.8 - Voltage regulator (LTC)

This notebook demonstrates the operation of the multiconductor power flow with **load tap changer control** through transformer tap changers. The controller supports both individual phase control as well as gang-control. The proof is based on the tests defined in the WP1.8 specification document. 

## Remarks about this controller 

The controller is generally designed and tested for 3-phased 2-winding transformers with tap at the secondary (low-voltage) side. In contrast to the LDC controller (WP 1.7), the LTC controller does not support locked_forward and locked_reverse modes, but always acts bidirectionally. The goal of the control action is to stabilize the voltage at the secondary transformer terminal. In phase control mode, the tap settings of the individual phases are adjusted to meet the target voltage bandwidth. In gang control mode, all phases are adjusted to the same tap position depending on the phase voltage with highest deviation from the target voltage bandwith. 

The latter case however may be problematic since the gang control might bring one phase voltage into the allowed bandwidth, but throw another phase voltage outside of it. The risk for this is especially high if voltages are highly asymmetric. In such a case, the controller would start to oscillate between two states where one or two phase voltages would be out of bandwith. This is different to the LDC controller from WP 1.7, which can also oscillate, but only if the voltage bandwidth is too narrow. For the LTC, oscillation may occur because of high asymmetry even if the bandwidth is broad. 

In order to address this, a special controller feature for oscillation detection was added. It can be optionally activated. If active, the controller will converge when oscillation is detected, and a warning will be printed to the console. This however leaves some of the phase voltages outside the allowed bandwidth. 

Apart from this, there are two notable reasons why the controller might be unable to stabilize the voltage within the bandwidth:
(1) Limitation of the tap setting. If this is the case, the controller will drive the tap to the minimum or maximum position. 
(2) Too narrow bandwidth selection. If this is the case (especially if the target voltage bandwith is lower than the effect of one tap position change), the controller might start to oscillate between two tap settings, and thus not converge. 

All in all, care needs to be taken for the bandwidth parameter to avoid non-convergence. 


## Basic imports

Here, the basic needed functions are imported and the test grids are loaded. 

In [1]:
import multiconductor as mc
import multiconductor.test.testing_toolbox
from multiconductor.pycci.std_types import create_std_type
from multiconductor.control.load_tap_changer_control import add_load_tap_changer_control, check_ltc_stable_state
from pandapower.control import run_control
import warnings
import pandas as pd 
from math import sqrt
import numpy as np
import copy 

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option('display.float_format', '{:.6f}'.format) 


In [2]:
grids = mc.test.testing_toolbox.load_all_test_grids("wp1_8_ltc")

Note: the following code will produce a warning because one of the grids is used for demonstrating the oscillation detection feature. 

In [3]:
l = []
for name, net in grids.items():
    net = copy.deepcopy(net)
    mc.run_pf(net, run_control=True)
    c = multiconductor.test.testing_toolbox.compare_to_dss(net)
    l.append(c)
    print('.', end='')
print()    
pd.DataFrame(l, index=grids.keys()).sort_index(key=lambda x: x.str.len())

............


,max_vm_pu_diff,max_va_degree_diff,max_i_from_ka_diff,max_i_to_ka_diff,max_p_from_mw_diff,max_p_to_mw_diff,max_q_from_mvar_diff,max_q_to_mvar_diff
twenty_bus_system_overvoltage_gang_controlled_LTC.pkl,0.000003,0.000662,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
four_bus_system_symmetric_load_gang_controlled_LTC.pkl,0.000036,0.004728,0.000025,0.000025,0.000016,0.000008,0.000058,0.000029
four_bus_system_symmetric_sgen_gang_controlled_LTC.pkl,0.000003,0.003677,0.000002,0.000002,0.000001,0.000001,0.000005,0.000002
twenty_bus_system_overvoltage_phase_controlled_LTC.pkl,0.000003,0.000662,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
twenty_bus_system_undervoltage_gang_controlled_LTC.pkl,0.000015,0.002120,0.000002,0.000002,0.000000,0.000000,0.000000,0.000000
four_bus_system_asymmetric_load_gang_controlled_LTC.pkl,0.000014,0.004442,0.000009,0.000009,0.000003,0.000004,0.000012,0.000017
four_bus_system_asymmetric_sgen_gang_controlled_LTC.pkl,0.000018,0.006128,0.000013,0.000013,0.000014,0.000013,0.000047,0.000033
four_bus_system_symmetric_load_phase_controlled_LTC.pkl,0.000036,0.004728,0.000025,0.000025,0.000016,0.000008,0.000058,0.000029
four_bus_system_symmetric_sgen_phase_controlled_LTC.pkl,0.000003,0.003677,0.000002,0.000002,0.000001,0.000001,0.000005,0.000002
twenty_bus_system_undervoltage_phase_controlled_LTC.pkl,0.000015,0.002123,0.000002,0.000002,0.000000,0.000000,0.000000,0.000000


## Controller validation function

In order to validate the controller operation, the function "check_ldc_stable_state" was designed. It checks if the controller was successful in stabilizing the voltage at the controlled bus within the given voltage bandwidth, or otherwise would have used maximum or minimum tap changer settings. The latter is an allowed state for cases in which the controller is unable to maintain the voltage bandwidth given the limitations of the tap changer. 
In case the controller is in gang operation mode, the function will also check if all tap settings are equal. 
The following code demonstrates the function. 

In [4]:
net = copy.deepcopy(grids["four_bus_system_symmetric_load_phase_controlled_LTC.pkl"])

print(f"grid contains {len(net.controller)} controller(s):")
for o in net.controller.object:
    print("    ", o)

print()
print("--- Run without controllers ---")
mc.run_pf(net, run_control=False)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="phase")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

print()
print("--- Run with controllers and re-check ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="phase")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))


grid contains 1 controller(s):
     LoadTapChangerControl of trafo1ph 0

--- Run without controllers ---
Check passed: False
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9555513031318296, 0.955551303131829, 0.9555513031318296]

--- Run with controllers and re-check ---
Check passed: True
Tap settings: [12, 12, 12]
Voltages at controlled bus: [1.030520714215941, 1.030520714790198, 1.0305207147990805]


If the oscillation detection feature is activated and an oscillation indeed was detected during the last power flow run, the function will always return True. This is demonstrated as follows.

In [5]:
net = copy.deepcopy(grids["four_bus_system_asymmetric_load_gang_controlled_oscillationdetect_LTC.pkl"])

print()
print("--- Run without controllers ---")
mc.run_pf(net, run_control=False)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="gang")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

print()
print("--- Run with controllers and re-check ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="gang")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))


--- Run without controllers ---
Check passed: False
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9862212761676986, 0.9519324234870837, 1.0031942161481964]

--- Run with controllers and re-check ---
Check passed: True
Tap settings: [11, 11, 11]
Voltages at controlled bus: [1.0551619124954648, 1.0237492585080545, 1.0723139753965343]


As can be seen, the oscillation was detected and the controller converged with two phase voltages outside the allowed voltage bandwidth of 1.03..1.05 p.u. 

In order to create a volt/var controller one can use the function add_load_tap_changer_control:

In [6]:
net = copy.deepcopy(grids["four_bus_system_symmetric_load_phase_controlled_LTC.pkl"])
# delete existing controllers
net.controller.drop(net.controller.index, inplace=True)

# Setup controller parameters and add controller
# Note: controlled bus nominal voltage is 16 kV
add_load_tap_changer_control(net, trafo_top_level_index=0, mode="phase", vm_lower_pu=1.03, vm_upper_pu=1.05, detect_oscillation=False)

print(f"grid contains {len(net.controller)} controller(s):")
for o in net.controller.object:
    print("    ", o)

mc.run_pf(net, run_control=True)
check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode='phase')
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

INFO:pandapower.control.util.auxiliary:Creating controller 0 of type <class 'multiconductor.control.load_tap_changer_control.LoadTapChangerControl'> 


grid contains 1 controller(s):
     LoadTapChangerControl of trafo1ph 0
Tap settings: [12, 12, 12]
Voltages at controlled bus: [1.030520714215941, 1.030520714790198, 1.0305207147990805]


# Tests with 4-bus topologies 


## Test 1: Symmetrical load at Bus3, LTC controlling Bus1 in phase mode
Controller to stabilize controlled bus voltage within voltage band of 1.03..1.05 p.U.

In [7]:
net = grids["four_bus_system_symmetric_load_phase_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="phase")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9555513031318296, 0.955551303131829, 0.9555513031318296]

--- Post control results ---
Check passed: True
Tap settings: [12, 12, 12]
Voltages at controlled bus: [1.030520714215941, 1.030520714790198, 1.0305207147990805]
Max overall vm_pu difference: 3.588893946815386e-05
Max overall va_degree difference: 0.004727968843869945
Max overall i_from_ka difference: 2.516477983927068e-05
Max overall i_to_ka difference: 2.516477983927068e-05
Max overall p_from_mw difference: 1.6433194214293678e-05
Max overall p_to_mw difference: 8.218917225910616e-06
Max overall q_from_mvar difference: 5.7501182392005745e-05
Max overall q_to_mvar difference: 2.8751212935329562e-05

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000    -83.212735       0.000000
      1      1.000

## Test 2: Symmetrical load at Bus3, LTC controlling Bus1 in gang mode
Controller to stabilize controlled bus voltage within voltage band of 1.03..1.05 p.U.

In [8]:
net = grids["four_bus_system_symmetric_load_gang_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="gang")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9555513031318296, 0.955551303131829, 0.9555513031318296]

--- Post control results ---
Check passed: True
Tap settings: [12, 12, 12]
Voltages at controlled bus: [1.030520714215941, 1.030520714790198, 1.0305207147990805]
Max overall vm_pu difference: 3.588893946815386e-05
Max overall va_degree difference: 0.004727968843869945
Max overall i_from_ka difference: 2.516477983927068e-05
Max overall i_to_ka difference: 2.516477983927068e-05
Max overall p_from_mw difference: 1.6433194214293678e-05
Max overall p_to_mw difference: 8.218917225910616e-06
Max overall q_from_mvar difference: 5.7501182392005745e-05
Max overall q_to_mvar difference: 2.8751212935329562e-05

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000    -83.212735       0.000000
      1      1.000

## Test 3: Asymmetrical load at Bus3, LTC controlling Bus1 in phase mode
Controller to stabilize controlled bus voltage within voltage band of 1.03..1.05 p.U.

In [9]:
net = grids["four_bus_system_asymmetric_load_phase_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="phase")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9862212761676986, 0.9519324234870837, 1.0031942161481964]

--- Post control results ---
Check passed: True
Tap settings: [8, 12, 5]
Voltages at controlled bus: [1.0358249346707737, 1.0306374789669748, 1.0347136125434968]
Max overall vm_pu difference: 4.058997373146411e-05
Max overall va_degree difference: 0.008787539592148619
Max overall i_from_ka difference: 3.291166177321614e-05
Max overall i_to_ka difference: 3.291166177321614e-05
Max overall p_from_mw difference: 2.6451692469287025e-05
Max overall p_to_mw difference: 1.4337677881393063e-05
Max overall q_from_mvar difference: 0.00013565541615068177
Max overall q_to_mvar difference: 9.325636509283086e-05

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000   -108.667252    -108.658562
      1      1.00

## Test 4: Asymmetrical load at Bus3, LTC controlling Bus1 in gang mode
Controller to stabilize controlled bus voltage within voltage band of 1.03..1.05 p.U.

In [10]:
net = grids["four_bus_system_asymmetric_load_gang_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="gang")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9840120556469322, 0.9801704532698894, 0.9970818157080112]

--- Post control results ---
Check passed: True
Tap settings: [8, 8, 8]
Voltages at controlled bus: [1.0344488362901423, 1.030598713753068, 1.0473544615335941]
Max overall vm_pu difference: 1.4193162593989506e-05
Max overall va_degree difference: 0.004441532020187822
Max overall i_from_ka difference: 8.545895246764301e-06
Max overall i_to_ka difference: 8.545895246764301e-06
Max overall p_from_mw difference: 2.7777719209254315e-06
Max overall p_to_mw difference: 4.394835943166697e-06
Max overall q_from_mvar difference: 1.2338000986100717e-05
Max overall q_to_mvar difference: 1.6580790057785233e-05

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000    -96.114759       0.000000
      1      1.000

## Test 5: Symmetrical sgen at Bus3, LTC controlling Bus1 in phase mode
Controller to stabilize controlled bus voltage within voltage band of 0.95..0.97 p.U.

In [11]:
net = grids["four_bus_system_symmetric_sgen_phase_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[0.95,0.97], mode="phase")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9992967067519243, 0.999296706751924, 0.9992967067519243]

--- Post control results ---
Check passed: True
Tap settings: [-5, -5, -5]
Voltages at controlled bus: [0.9673649102383349, 0.9673649102383346, 0.9673649102383348]
Max overall vm_pu difference: 3.4167234094217847e-06
Max overall va_degree difference: 0.003677065945026925
Max overall i_from_ka difference: 2.1070549074408973e-06
Max overall i_to_ka difference: 2.1070549074408973e-06
Max overall p_from_mw difference: 1.3268514322817282e-06
Max overall p_to_mw difference: 6.65821977108294e-07
Max overall q_from_mvar difference: 4.627760763320499e-06
Max overall q_to_mvar difference: 2.3141483771471627e-06

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000      0.000000       0.000000
      1      1.

## Test 6: Symmetrical sgen at Bus3, LTC controlling Bus1 in gang mode
Controller to stabilize controlled bus voltage within voltage band of 0.95..0.97 p.U.

In [12]:
net = grids["four_bus_system_symmetric_sgen_gang_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[0.95,0.97], mode="gang")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9992967067519243, 0.999296706751924, 0.9992967067519243]

--- Post control results ---
Check passed: True
Tap settings: [-5, -5, -5]
Voltages at controlled bus: [0.9673649102383349, 0.9673649102383346, 0.9673649102383348]
Max overall vm_pu difference: 3.4167234094217847e-06
Max overall va_degree difference: 0.003677065945026925
Max overall i_from_ka difference: 2.1070549074408973e-06
Max overall i_to_ka difference: 2.1070549074408973e-06
Max overall p_from_mw difference: 1.3268514322817282e-06
Max overall p_to_mw difference: 6.65821977108294e-07
Max overall q_from_mvar difference: 4.627760763320499e-06
Max overall q_to_mvar difference: 2.3141483771471627e-06

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000      0.000000       0.000000
      1      1.

## Test 7: Asymmetrical sgen at Bus3, LTC controlling Bus1 in phase mode
Controller to stabilize controlled bus voltage within voltage band of 0.95..0.97 p.U.

In [13]:
net = grids["four_bus_system_asymmetric_sgen_phase_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[0.95,0.97], mode="phase")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.988868226620235, 0.9567711871210179, 0.9946993920221362]

--- Post control results ---
Check passed: True
Tap settings: [-4, 0, -4]
Voltages at controlled bus: [0.9637419183281228, 0.9568822577638727, 0.9698460018887425]
Max overall vm_pu difference: 3.844651212703276e-05
Max overall va_degree difference: 0.006106157045934424
Max overall i_from_ka difference: 5.29477395718736e-05
Max overall i_to_ka difference: 5.29477395718736e-05
Max overall p_from_mw difference: 4.57280453760589e-05
Max overall p_to_mw difference: 4.586175086718214e-05
Max overall q_from_mvar difference: 0.00020393119471773247
Max overall q_to_mvar difference: 0.00015520230075782848

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000    112.607792     112.601630
      1      1.000000

## Test 8: Asymmetrical sgen at Bus3, LTC controlling Bus1 in gang mode
Controller to stabilize controlled bus voltage within voltage band of 0.95..0.97 p.U.

In [14]:
net = grids["four_bus_system_asymmetric_sgen_gang_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[0.95,0.97], mode="gang")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9932805260359572, 0.9775195346456124, 0.9851462911226869]

--- Post control results ---
Check passed: True
Tap settings: [-4, -4, -4]
Voltages at controlled bus: [0.9683071206884377, 0.9521875695918188, 0.9602371098848074]
Max overall vm_pu difference: 1.7808207509073526e-05
Max overall va_degree difference: 0.00612797967107781
Max overall i_from_ka difference: 1.2550496268026734e-05
Max overall i_to_ka difference: 1.2550496268026734e-05
Max overall p_from_mw difference: 1.356174679223443e-05
Max overall p_to_mw difference: 1.3439938031822862e-05
Max overall q_from_mvar difference: 4.67154308096962e-05
Max overall q_to_mvar difference: 3.284266019676352e-05

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000     74.460620       0.000000
      1      1.0

# Tests with 20-bus system with LDC controlled transformer between external grid and low voltage level

## Test 9: Undervoltage situation with LDC controlling Bus1 in phase mode
Controller to stabilize controlled bus voltage within voltage band of 1.03..1.05 p.U.

In [15]:
net = grids["twenty_bus_system_undervoltage_phase_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="phase")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9859073654378053, 0.9854646875672768, 0.9910580728397717]

--- Post control results ---
Check passed: True
Tap settings: [8, 8, 7]
Voltages at controlled bus: [1.0354057071612304, 1.034864915795496, 1.0345482313252394]
Max overall vm_pu difference: 1.54834202301668e-05
Max overall va_degree difference: 0.0021230763299016076
Max overall i_from_ka difference: 2.117486840563787e-06
Max overall i_to_ka difference: 2.117457945122192e-06
Max overall p_from_mw difference: 1.4556430582429725e-07
Max overall p_to_mw difference: 1.222525749938752e-07
Max overall q_from_mvar difference: 4.043047448082371e-07
Max overall q_to_mvar difference: 4.509433968666321e-07

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000     -2.874733      -2.871852
      1      1.000000

## Test 10: Undervoltage situation with LDC controlling Bus1 in gang mode
Controller to stabilize controlled bus voltage within voltage band of 1.03..1.05 p.U.

In [16]:
net = grids["twenty_bus_system_undervoltage_gang_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[1.03,1.05], mode="gang")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [0.9859073654378053, 0.9854646875672768, 0.9910580728397717]

--- Post control results ---
Check passed: True
Tap settings: [8, 8, 8]
Voltages at controlled bus: [1.0354235557153144, 1.034879730326556, 1.040724758287715]
Max overall vm_pu difference: 1.546452470391202e-05
Max overall va_degree difference: 0.0021204488404222843
Max overall i_from_ka difference: 2.1149276562548724e-06
Max overall i_to_ka difference: 2.1148987895958093e-06
Max overall p_from_mw difference: 1.4642641035095272e-07
Max overall p_to_mw difference: 1.2320073768112838e-07
Max overall q_from_mvar difference: 4.0251288484227654e-07
Max overall q_to_mvar difference: 4.489794294126881e-07

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000     -2.568316       0.000000
      1      1.00

## Test 11: Overvoltage situation with LDC controlling Bus1 in phase mode
Controller to stabilize controlled bus voltage within voltage band of 0.95..0.97 p.U.

In [17]:
net = grids["twenty_bus_system_overvoltage_phase_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[0.95,0.97], mode="phase")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0004267160808904, 1.0005541376434708, 0.9998666871303947]

--- Post control results ---
Check passed: True
Tap settings: [-5, -5, -5]
Voltages at controlled bus: [0.9691201138317577, 0.9692625116659156, 0.9686063137945689]
Max overall vm_pu difference: 2.639001826398868e-06
Max overall va_degree difference: 0.0006621957383301513
Max overall i_from_ka difference: 3.1048977852665693e-07
Max overall i_to_ka difference: 3.1049383264480923e-07
Max overall p_from_mw difference: 2.7125549546119787e-08
Max overall p_to_mw difference: 2.8048831217722392e-08
Max overall q_from_mvar difference: 7.444489351360328e-08
Max overall q_to_mvar difference: 7.458641170399877e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000   -155.174060       0.000000
      1      

## Test 12: Overvoltage situation with LDC controlling Bus1 in gang mode
Controller to stabilize controlled bus voltage within voltage band of 0.95..0.97 p.U.

In [18]:
net = grids["twenty_bus_system_overvoltage_gang_controlled_LTC.pkl"]

print("--- Pre/Post control comparison ---")
print("--- Pre control results ---")
mc.run_pf(net)
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))
print("\n--- Post control results ---")
mc.run_pf(net, run_control=True)
passed = check_ltc_stable_state(net, controlled_trafo_index=0, voltage_bandwidth=[0.95,0.97], mode="gang")
print("Check passed: "+str(passed))
print("Tap settings: ", end='')
print(list(net.trafo1ph.loc[0]["tap_pos"][3:]))
print("Voltages at controlled bus: ", end='')
print(list(net.res_bus.loc[1]["vm_pu"][1:]))

c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)

print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
--- Pre control results ---
Tap settings: [0, 0, 0]
Voltages at controlled bus: [1.0004267160808904, 1.0005541376434708, 0.9998666871303947]

--- Post control results ---
Check passed: True
Tap settings: [-5, -5, -5]
Voltages at controlled bus: [0.9691201138317577, 0.9692625116659156, 0.9686063137945689]
Max overall vm_pu difference: 2.639001826398868e-06
Max overall va_degree difference: 0.0006621957383301513
Max overall i_from_ka difference: 3.1048977852665693e-07
Max overall i_to_ka difference: 3.1049383264480923e-07
Max overall p_from_mw difference: 2.7125549546119787e-08
Max overall p_to_mw difference: 2.8048831217722392e-08
Max overall q_from_mvar difference: 7.444489351360328e-08
Max overall q_to_mvar difference: 7.458641170399877e-08

bus results
             vm_pu_mc  vm_pu_dss  va_degree_mc  va_degree_dss
index phase                                                  
0     0      0.000000   0.000000   -155.174060       0.000000
      1      